In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/root/autodl-tmp/Qwen/Qwen2___5-3B-Instruct", trust_remote_code=True)

In [3]:
from datasets import Dataset

In [6]:
def prepare_datasets(train_path, val_path, tokenizer):
    """准备数据集"""
    train_ds = Dataset.from_csv(train_path)
    eval_ds = Dataset.from_csv(val_path)
    
    def apply_template(ds):
        messages = [
            {"role": "user", "content": ds["prompt"]},
            {"role": "assistant", "content": ds["response"]}
        ]
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": input_text}
    
    def tokenize_function(ds):
        tokenized = tokenizer(
            ds["text"], 
            truncation=True, 
            padding=True,
            # max_length=max_length
        )
        return tokenized
    
    # 应用模板和分词
    train_template_ds = train_ds.map(apply_template)
    eval_template_ds = eval_ds.map(apply_template)
    
    train_tk_ds = train_template_ds.map(tokenize_function, remove_columns=train_template_ds.column_names)
    eval_tk_ds = eval_template_ds.map(tokenize_function, remove_columns=eval_template_ds.column_names)
    
    return train_tk_ds, eval_tk_ds

In [7]:
train_tk_ds, eval_tk_ds = prepare_datasets('/root/autodl-fs/data/safety_aware_train_sft_data.csv',
                                           '/root/autodl-fs/data/safety_aware_val_sft_data.csv',
                                             tokenizer)

Map:   0%|          | 0/17165 [00:00<?, ? examples/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

Map:   0%|          | 0/17165 [00:00<?, ? examples/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

In [15]:
lengths = []
for ids in train_tk_ds['input_ids']:
    lengths.append(len(ids))

In [18]:
lengths.sort()

In [25]:
lengths[17100]

1678